# 02 · Pipeline ETL Medallion — DataRetail 360 (EP3)

**IDY1102 · Estructuras y Visualización de Datos · Evaluación Parcial 3**

| Campo | Detalle |
|---|---|
| **Propósito** | Extraer la fuente generada por `01_generar_fuente.ipynb` (ventas, inventario y maestros), cargarla en **Bronze**, limpiarla y validarla en **Silver** y construir el modelo dimensional en **Gold** |
| **Stack** | Python · Pandas · SQLAlchemy (`mssql+pyodbc`) · SQL Server Express · SSMS |
| **Base de datos** | `DataRetail360` con esquemas `bronze`, `silver`, `gold` y `audit` |
| **Trazabilidad** | `run_id` por ejecución · `archivo_origen` por fila · log en `logs/` · tabla `audit.log_ejecuciones` |

### Flujo
1. **Extracción**: verificación SHA-256 contra `manifest.json` y lectura de 90 CSV de ventas, 90 CSV de inventario y maestros v1/v2.
2. **Bronze**: copia fiel de la fuente (todo texto) + linaje.
3. **Silver**: reglas de calidad DAMA, cuarentena, deduplicación y seudonimización (Ley 21.719).
4. **Gold**: dimensiones con claves sustitutas, SCD 1/2/3, filas −1, 4 tablas de hechos (Ventas, Inventario, Marketing, Finanzas), PK y FK.
5. **Validación post-carga** y registro de auditoría.

**Requisitos:** ejecutar antes `01_generar_fuente.ipynb` y el script `sql/00_crear_base.sql` en SSMS. Paquetes: `pip install pandas sqlalchemy pyodbc`.
Ejecutar con **Run All** desde la carpeta `notebooks/`.

## 0. Parámetros

In [ ]:
from pathlib import Path
from datetime import datetime, date
import json, hashlib, logging, sys, time, uuid, urllib
import numpy as np
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.types import Integer, NVARCHAR, Date, DateTime, Numeric, Boolean

BASE       = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FUENTE     = BASE / "data" / "fuente"
DIR_VENTAS = FUENTE / "ventas"
DIR_INV    = FUENTE / "inventario"
DIR_M      = {"v1": FUENTE / "maestros" / "v1", "v2": FUENTE / "maestros" / "v2"}
DIR_LOGS   = BASE / "logs"
DIR_LOGS.mkdir(exist_ok=True)

# Conexión a SQL Server (misma configuración que ya funcionaba)
SERVIDOR   = r".\SQLEXPRESS"
BASE_DATOS = "DataRetail360"
DRIVER     = "ODBC Driver 17 for SQL Server"

# Vigencias del caso
FECHA_INICIO_PERIODO = "2025-01-01"   # inicio de la primera versión de los maestros
FECHA_FIN_PERIODO    = "2025-03-31"
FECHA_FIN_V1         = "2025-02-28"   # cierre de la versión anterior (SCD 2)
FECHA_V2             = "2025-03-01"   # entrada en vigencia de la 2.ª entrega de maestros
FECHA_ABIERTA        = "9999-12-31"   # fecha_fin de la versión vigente no se si funciona

RUN_ID      = datetime.now().strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:6]
FECHA_CARGA = datetime.now().replace(microsecond=0)
print("run_id:", RUN_ID, "| pandas", pd.__version__, "| base:", BASE)

## 1. Logging
Cada mensaje queda en la consola y en `logs/pipeline_<run_id>.log` con fecha, nivel y etapa. `RESUMEN` acumula los conteos que se guardan al final en `audit.log_ejecuciones`.

In [ ]:
LOG_FILE = DIR_LOGS / f"pipeline_{RUN_ID}.log"

class FiltroEtapa(logging.Filter):
    def filter(self, registro):
        if not hasattr(registro, "etapa"):
            registro.etapa = "GENERAL"
        return True

logger = logging.getLogger("dataretail360")
logger.setLevel(logging.INFO)
logger.handlers.clear()
logger.propagate = False
formato = logging.Formatter("%(asctime)s | %(levelname)-7s | %(etapa)-10s | %(message)s", "%Y-%m-%d %H:%M:%S")
for handler in (logging.StreamHandler(sys.stdout), logging.FileHandler(LOG_FILE, encoding="utf-8")):
    handler.setFormatter(formato)
    handler.addFilter(FiltroEtapa())
    logger.addHandler(handler)

def log(etapa):
    return logging.LoggerAdapter(logger, {"etapa": etapa})

RESUMEN = {"run_id": RUN_ID, "inicio": datetime.now().replace(microsecond=0), "estado": "EN CURSO"}

def fallar(etapa):
    """Registra el error con su traza y marca la ejecución como fallida."""
    log(etapa).exception(f"La etapa {etapa} falló; el pipeline se detiene")
    RESUMEN["estado"] = f"FALLIDA en {etapa}"

log("INICIO").info(f"Ejecución {RUN_ID} iniciada · log en {LOG_FILE.name}")

## 2. Conexión con SQLAlchemy
Se usa `create_engine` con el dialecto `mssql+pyodbc`. `fast_executemany=True` acelera la inserción de ~500.000 filas.

In [ ]:
def crear_motor(servidor=SERVIDOR, base=BASE_DATOS):
    params = urllib.parse.quote_plus(
        f"DRIVER={{{DRIVER}}};SERVER={servidor};DATABASE={base};"
        "Trusted_Connection=yes;TrustServerCertificate=yes;")
    return create_engine(f"mssql+pyodbc:///?odbc_connect={params}",
                         fast_executemany=True, connect_args={"timeout": 10})

def probar_conexion(motor):
    with motor.connect() as con:
        return con.execute(text("SELECT @@SERVERNAME, DB_NAME()")).one()

try:
    engine = crear_motor()
    servidor, base = probar_conexion(engine)
    log("CONEXION").info(f"Conectado a SQL Server {servidor} · base {base}")
except Exception:
    fallar("CONEXION")
    raise

### 2.1 Prueba de error controlado
Se intenta conectar a un servidor que no existe para demostrar que el error se captura y se registra en el log sin romper el notebook.

In [ ]:
try:
    probar_conexion(crear_motor(servidor=r".\SERVIDOR_INEXISTENTE"))
except Exception as e:
    log("CONEXION").error(f"Error controlado (prueba): {type(e).__name__}: {str(e)[:160]}")

## 3. Funciones de escritura y consulta
Cada capa se escribe dentro de **una sola transacción** (`engine.begin()`): si una tabla falla, la capa completa se revierte.

In [ ]:
ESQUEMAS = ("bronze", "silver", "gold", "audit")

def crear_esquemas():
    with engine.begin() as con:
        for e in ESQUEMAS:
            con.execute(text(f"IF SCHEMA_ID('{e}') IS NULL EXEC('CREATE SCHEMA {e}')"))
    log("BD").info(f"Esquemas verificados: {', '.join(ESQUEMAS)}")

def tipos_texto(df, largo=200):
    """Bronze guarda todo como texto; las columnas de linaje llevan su propio tipo."""
    tipos = {c: NVARCHAR(largo) for c in df.columns}
    if "fecha_carga" in df.columns:
        tipos["fecha_carga"] = DateTime()
    return tipos

def escribir_capa(esquema, tablas, etapa, borrar_antes=()):
    """tablas = {nombre: (DataFrame, dtype)}. Todo en una transacción."""
    t0 = time.perf_counter()
    with engine.begin() as con:
        for nombre in borrar_antes:                      # tablas con FK se borran en orden
            con.execute(text(f"DROP TABLE IF EXISTS {esquema}.{nombre}"))
        for nombre, (df, dtype) in tablas.items():
            t1 = time.perf_counter()
            df.to_sql(nombre, con=con, schema=esquema, if_exists="replace",
                      index=False, dtype=dtype, chunksize=20_000)
            log(etapa).info(f"{esquema}.{nombre}: {len(df):,} filas ({time.perf_counter() - t1:.1f}s)")
    log(etapa).info(f"Capa {esquema} escrita en {time.perf_counter() - t0:.1f}s")

def ejecutar(sentencias):
    with engine.begin() as con:
        for s in sentencias:
            con.execute(text(s))

def consultar(sql):
    with engine.connect() as con:
        return pd.read_sql(text(sql), con)

def a_fecha(serie):
    """'AAAA-MM-DD' → date (admite 9999-12-31, que no cabe en datetime64)."""
    return serie.map(lambda x: date.fromisoformat(x) if isinstance(x, str) and x else None)

## 4. Extracción
1. Se verifica la huella **SHA-256** de cada archivo contra `manifest.json`; si alguna no coincide, el pipeline se detiene.
2. Se leen los 90 CSV como texto (`dtype=str`) para no perder los valores mal escritos. Un archivo sin las columnas esperadas se omite y se registra.
3. Se leen los maestros v1 y v2.

In [ ]:
lg = log("EXTRACCION")
COLS_VENTAS = ["id_venta", "nro_boleta", "nro_linea", "fecha", "sku", "sucursal", "canal", "cantidad",
               "precio_unitario", "monto", "descuento", "cod_campana", "estado", "rut_cliente"]

def sha256(ruta):
    h = hashlib.sha256()
    with open(ruta, "rb") as f:
        for bloque in iter(lambda: f.read(1 << 20), b""):
            h.update(bloque)
    return h.hexdigest()

try:
    manifest = json.loads((FUENTE / "manifest.json").read_text(encoding="utf-8"))
    distintos = [r for r, h in manifest["archivos"].items() if sha256(FUENTE / r) != h]
    if distintos:
        raise ValueError(f"{len(distintos)} archivos no coinciden con manifest.json: {distintos[:5]}")
    lg.info(f"Integridad OK: {len(manifest['archivos'])} archivos verificados con SHA-256")

    partes, omitidos = [], []
    for archivo in sorted(DIR_VENTAS.glob("ventas_*.csv")):
        df = pd.read_csv(archivo, dtype=str, keep_default_na=False)
        faltan = set(COLS_VENTAS) - set(df.columns)
        if faltan:
            lg.error(f"{archivo.name} omitido: faltan columnas {sorted(faltan)}")
            omitidos.append(archivo.name)
            continue
        partes.append(df[COLS_VENTAS].assign(archivo_origen=archivo.name))
    ventas_raw = pd.concat(partes, ignore_index=True)
    lg.info(f"Ventas: {len(partes)} archivos leídos, {len(omitidos)} omitidos, {len(ventas_raw):,} filas")

    COLS_INV = ["fecha", "cod_sucursal", "sku", "cantidad_stock", "stock_minimo"]
    partes_inv = []
    for archivo in sorted(DIR_INV.glob("inventario_*.csv")):
        df = pd.read_csv(archivo, dtype=str, keep_default_na=False)
        faltan = set(COLS_INV) - set(df.columns)
        if faltan:
            lg.error(f"{archivo.name} omitido: faltan columnas {sorted(faltan)}")
            continue
        partes_inv.append(df[COLS_INV].assign(archivo_origen=archivo.name))
    if not partes_inv:
        raise FileNotFoundError("No hay archivos de inventario: ejecutar primero 01_generar_fuente.ipynb")
    inventario_raw = pd.concat(partes_inv, ignore_index=True)
    lg.info(f"Inventario: {len(partes_inv)} archivos leídos, {len(inventario_raw):,} filas")

    def leer_maestro(nombre):
        filas = []
        for version, carpeta in DIR_M.items():
            ruta = carpeta / nombre
            df = (pd.read_json(ruta, dtype=False) if ruta.suffix == ".json"
                  else pd.read_csv(ruta, dtype=str, keep_default_na=False))
            filas.append(df.astype(str).assign(version=version, archivo_origen=f"maestros/{version}/{nombre}"))
        return pd.concat(filas, ignore_index=True)

    productos_raw  = leer_maestro("productos.json")
    sucursales_raw = leer_maestro("sucursales.csv")
    clientes_raw   = leer_maestro("clientes.csv")
    campanas_raw   = leer_maestro("campanas.csv")
    for n, d in [("productos", productos_raw), ("sucursales", sucursales_raw),
                 ("clientes", clientes_raw), ("campañas", campanas_raw)]:
        lg.info(f"Maestro {n}: {len(d):,} filas (v1 + v2)")
except Exception:
    fallar("EXTRACCION")
    raise

RESUMEN["filas_fuente"] = len(ventas_raw)
print(f"Filas antes de limpieza: {len(ventas_raw)}")
print(f"Filas de inventario antes de limpieza: {len(inventario_raw)}")

## 5. Capa Bronze
Los datos se guardan **tal como llegaron** (texto), más `archivo_origen`, `run_id` y `fecha_carga`. Permite reprocesar desde cero y auditar qué llegó exactamente.

In [ ]:
def con_linaje(df):
    return df.assign(run_id=RUN_ID, fecha_carga=FECHA_CARGA)

try:
    crear_esquemas()
    bronze = {
        "ventas_raw":     con_linaje(ventas_raw),
        "inventario_raw": con_linaje(inventario_raw),
        "productos_raw":  con_linaje(productos_raw),
        "sucursales_raw": con_linaje(sucursales_raw),
        "clientes_raw":   con_linaje(clientes_raw),
        "campanas_raw":   con_linaje(campanas_raw),
    }
    escribir_capa("bronze", {k: (v, tipos_texto(v)) for k, v in bronze.items()}, "BRONZE")
except Exception:
    fallar("BRONZE")
    raise
RESUMEN["filas_bronze"] = len(bronze["ventas_raw"])

## 6. Capa Silver: ventas
Cada regla se asocia a una dimensión de calidad DAMA y queda registrada con su número de filas en `silver.reglas_calidad`.
Ninguna fila inválida se elimina en silencio: va a `silver.cuarentena_ventas` con su motivo y archivo de origen.

| Tipo de regla | Acción |
|---|---|
| Se puede corregir (SKU, canal, fecha con `/`, RUT, monto negativo) | Se corrige |
| Valor que no existe en el maestro (sucursal, SKU, campaña) | Se conserva la fila y en Gold toma la clave −1 |
| No se puede corregir (fecha imposible, monto no numérico) | Cuarentena |
| Línea duplicada | Se conserva la primera aparición |

In [ ]:
lg = log("SILVER")
reglas = []
def regla(nombre, dama, filas, accion, tabla="ventas"):
    reglas.append({"tabla": tabla, "regla": nombre, "dimension_dama": dama, "filas": int(filas), "accion": accion})
    log("SILVER").info(f"[{tabla}] Regla {nombre:<24} [{dama:<12}] {int(filas):>7,} filas → {accion}")

try:
    s = ventas_raw.copy()

    # --- Unicidad ---------------------------------------------------------
    dup = s.duplicated("id_venta", keep="first")
    regla("linea_duplicada", "Unicidad", dup.sum(), "Se conserva la primera aparición")
    s = s[~dup].copy()

    # --- Consistencia -----------------------------------------------------
    sku_limpio = s["sku"].str.strip().str.upper()
    regla("sku_formato", "Consistencia", (s["sku"] != sku_limpio).sum(), "strip() + upper()")
    s["sku"] = sku_limpio

    MAPEO_CANAL = {"Tienda fisica": "Tienda física", "ecommerce": "E-commerce", "APP": "App"}
    regla("canal_otra_grafia", "Consistencia", s["canal"].isin(MAPEO_CANAL).sum(), "Mapeo al nombre oficial")
    s["canal"] = s["canal"].replace(MAPEO_CANAL)

    con_slash = s["fecha"].str.contains("/", regex=False)
    regla("fecha_con_slash", "Validez", con_slash.sum(), "Reemplazo de '/' por '-'")
    s["fecha"] = s["fecha"].str.replace("/", "-", regex=False)

    sin_puntos = s["rut_cliente"].ne("") & ~s["rut_cliente"].str.contains(".", regex=False)
    regla("rut_sin_puntos", "Consistencia", sin_puntos.sum(), "Se normaliza antes del hash")

    # --- Validez (lo que no se puede corregir va a cuarentena) ------------
    fecha    = pd.to_datetime(s["fecha"], format="%Y-%m-%d", errors="coerce")
    monto    = pd.to_numeric(s["monto"], errors="coerce")
    cantidad = pd.to_numeric(s["cantidad"], errors="coerce")
    precio   = pd.to_numeric(s["precio_unitario"], errors="coerce")
    desc     = pd.to_numeric(s["descuento"], errors="coerce")
    fuera    = fecha.notna() & ~fecha.between(FECHA_INICIO_PERIODO, FECHA_FIN_PERIODO)

    condiciones = {
        "fecha_imposible":        fecha.isna(),
        "fecha_fuera_de_periodo": fuera,
        "monto_no_numerico":      monto.isna(),
        "cantidad_invalida":      cantidad.isna() | (cantidad <= 0),
        "precio_invalido":        precio.isna() | (precio <= 0),
        "descuento_invalido":     desc.isna() | (desc < 0),
        "canal_invalido":         ~s["canal"].isin(["Tienda física", "E-commerce", "App"]),
        "estado_invalido":        ~s["estado"].isin(["OK", "DEVUELTA"]),
    }
    motivo = pd.Series("", index=s.index)
    for nombre, cond in condiciones.items():          # se guarda el primer motivo encontrado
        motivo = motivo.mask(cond & motivo.eq(""), nombre)
        regla(nombre, "Validez", cond.sum(), "Cuarentena")

    cuarentena_ventas = s[motivo.ne("")].assign(motivo=motivo[motivo.ne("")], run_id=RUN_ID, fecha_carga=FECHA_CARGA)
    validas = motivo.eq("")
    s = s[validas].copy()
    fecha, monto, cantidad, precio, desc = (x[validas] for x in (fecha, monto, cantidad, precio, desc))

    negativos = monto < 0
    regla("monto_negativo", "Validez", negativos.sum(), "Valor absoluto")
    monto = monto.abs()
    distinto = monto != precio * cantidad
    regla("monto_distinto_calculo", "Exactitud", distinto.sum(), "Se usa precio × cantidad")

    # --- Integridad con los maestros (la fila se conserva, en Gold toma clave −1)
    sucursales_validas = set(sucursales_raw["sucursal"])
    skus_validos       = set(productos_raw["sku"])
    campanas_validas   = set(campanas_raw["cod_campana"])
    regla("sucursal_vacia_o_inexistente", "Completitud", (~s["sucursal"].isin(sucursales_validas)).sum(), "Clave −1 en Gold")
    regla("sku_inexistente", "Consistencia", (~s["sku"].isin(skus_validos)).sum(), "Clave −1 en Gold")
    camp_inex = s["cod_campana"].ne("") & ~s["cod_campana"].isin(campanas_validas)
    regla("campana_inexistente", "Consistencia", camp_inex.sum(), "Se asigna 'Sin campaña'")
    s.loc[camp_inex, "cod_campana"] = ""
except Exception:
    fallar("SILVER")
    raise

### 6.1 Seudonimización (Ley 21.719)
El RUT se normaliza (sin puntos, en mayúscula) y se reemplaza por su **hash SHA-256**. Nombre, correo y teléfono se descartan: no llegan a Silver ni a Gold.

In [ ]:
def normalizar_rut(rut):
    return str(rut).replace(".", "").replace(" ", "").upper()

def seudonimizar(rut):
    if pd.isna(rut) or str(rut).strip() == "":
        return ""                                     # cliente no identificado → clave −1 en Gold
    return hashlib.sha256(normalizar_rut(rut).encode("utf-8")).hexdigest()

silver_ventas = pd.DataFrame({
    "id_venta":        pd.to_numeric(s["id_venta"]).astype("int64"),
    "nro_boleta":      pd.to_numeric(s["nro_boleta"]).astype("int64"),
    "nro_linea":       pd.to_numeric(s["nro_linea"]).astype("int64"),
    "fecha":           fecha,
    "sku":             s["sku"],
    "sucursal":        s["sucursal"],
    "canal":           s["canal"],
    "cantidad":        cantidad.astype("int64"),
    "precio_unitario": precio.astype("int64"),
    "monto":           monto.astype("int64"),
    "descuento":       desc.astype("int64"),
    "cod_campana":     s["cod_campana"],
    "estado":          s["estado"],
    "id_cliente":      s["rut_cliente"].map(seudonimizar),   # el RUT no se guarda
    "archivo_origen":  s["archivo_origen"],
    "run_id":          RUN_ID,
    "fecha_carga":     FECHA_CARGA,
}).reset_index(drop=True)

reglas_calidad = pd.DataFrame(reglas).assign(run_id=RUN_ID)
RESUMEN["filas_silver"] = len(silver_ventas)
RESUMEN["filas_cuarentena"] = len(cuarentena_ventas)
print(f"Filas después de limpieza: {len(silver_ventas)}")
print(f"Filas en cuarentena: {len(cuarentena_ventas)}")
display(reglas_calidad[reglas_calidad["filas"] > 0])

### 6.2 Silver: inventario
Mismas reglas que en ventas, adaptadas a la foto diaria de stock. El grano es **fecha × sucursal × SKU**, por lo que esa combinación debe ser única.

In [ ]:
lg = log("SILVER")
try:
    inv = inventario_raw.copy()
    sku_limpio = inv["sku"].str.strip().str.upper()
    regla("sku_formato", "Consistencia", (inv["sku"] != sku_limpio).sum(), "strip() + upper()", tabla="inventario")
    inv["sku"] = sku_limpio

    dup = inv.duplicated(["fecha", "cod_sucursal", "sku"], keep="first")
    regla("fila_duplicada", "Unicidad", dup.sum(), "Se conserva la primera aparición", tabla="inventario")
    inv = inv[~dup].copy()

    fecha_inv = pd.to_datetime(inv["fecha"], format="%Y-%m-%d", errors="coerce")
    stock     = pd.to_numeric(inv["cantidad_stock"], errors="coerce")
    minimo    = pd.to_numeric(inv["stock_minimo"], errors="coerce")
    condiciones_inv = {
        "fecha_invalida":         fecha_inv.isna() | ~fecha_inv.between(FECHA_INICIO_PERIODO, FECHA_FIN_PERIODO),
        "cantidad_stock_vacia":   stock.isna(),
        "cantidad_stock_negativa": stock < 0,
        "stock_minimo_invalido":  minimo.isna() | (minimo < 0),
    }
    motivo_inv = pd.Series("", index=inv.index)
    for nombre, cond in condiciones_inv.items():
        motivo_inv = motivo_inv.mask(cond & motivo_inv.eq(""), nombre)
        regla(nombre, "Completitud" if "vacia" in nombre else "Validez", cond.sum(), "Cuarentena", tabla="inventario")
    cuarentena_inventario = inv[motivo_inv.ne("")].assign(motivo=motivo_inv[motivo_inv.ne("")],
                                                          run_id=RUN_ID, fecha_carga=FECHA_CARGA)
    ok = motivo_inv.eq("")
    regla("sku_inexistente", "Consistencia", (~inv.loc[ok, "sku"].isin(set(productos_raw["sku"]))).sum(),
          "Clave −1 en Gold", tabla="inventario")
    regla("sucursal_inexistente", "Consistencia", (~inv.loc[ok, "cod_sucursal"].isin(set(sucursales_raw["cod_sucursal"]))).sum(),
          "Clave −1 en Gold", tabla="inventario")

    silver_inventario = pd.DataFrame({
        "fecha":          fecha_inv[ok],
        "cod_sucursal":   inv.loc[ok, "cod_sucursal"],
        "sku":            inv.loc[ok, "sku"],
        "cantidad_stock": stock[ok].astype("int64"),
        "stock_minimo":   minimo[ok].astype("int64"),
        "archivo_origen": inv.loc[ok, "archivo_origen"],
        "run_id":         RUN_ID,
        "fecha_carga":    FECHA_CARGA,
    }).reset_index(drop=True)
    reglas_calidad = pd.DataFrame(reglas).assign(run_id=RUN_ID)
except Exception:
    fallar("SILVER")
    raise
print(f"Filas de inventario después de limpieza: {len(silver_inventario)}")
print(f"Filas de inventario en cuarentena: {len(cuarentena_inventario)}")

## 7. Capa Silver: maestros
- **Productos y sucursales**: se tipan; se conservan las dos versiones (v1, v2), insumo de los SCD.
- **Clientes**: se seudonimiza el RUT, se eliminan los duplicados de la v1 (mismo RUT, correo con otra grafía) y se descartan nombre, correo y teléfono. Las correcciones de contacto (SCD Tipo 1) no generan versiones porque esos campos no se conservan.

In [ ]:
lg = log("SILVER")
try:
    silver_productos = productos_raw.drop(columns="archivo_origen").astype(
        {"precio_lista": "int64", "costo_unitario": "int64"})
    silver_sucursales = sucursales_raw.drop(columns="archivo_origen").astype(
        {"latitud": "float64", "longitud": "float64"})
    silver_campanas = campanas_raw.drop(columns="archivo_origen").astype({"pct_descuento": "float64"})

    cli = clientes_raw.copy()
    cli["id_cliente"] = cli["rut_cliente"].map(seudonimizar)
    antes = len(cli)
    cli = cli.drop_duplicates(["version", "id_cliente"], keep="first")
    regla("cliente_duplicado", "Unicidad", antes - len(cli), "Se conserva el primero", tabla="clientes")
    silver_clientes = cli[["id_cliente", "segmento", "comuna", "region", "fecha_registro", "version"]].reset_index(drop=True)
    lg.info("Datos personales descartados de clientes: rut_cliente, nombre, email, telefono")
    reglas_calidad = pd.DataFrame(reglas).assign(run_id=RUN_ID)

    lin = {"run_id": RUN_ID, "fecha_carga": FECHA_CARGA}
    escribir_capa("silver", {
        "ventas":            (silver_ventas, {"fecha": Date(), "sku": NVARCHAR(20), "sucursal": NVARCHAR(50),
                                              "canal": NVARCHAR(30), "cod_campana": NVARCHAR(20), "estado": NVARCHAR(20),
                                              "id_cliente": NVARCHAR(64), "archivo_origen": NVARCHAR(60),
                                              "run_id": NVARCHAR(40), "fecha_carga": DateTime()}),
        "cuarentena_ventas": (cuarentena_ventas, tipos_texto(cuarentena_ventas)),
        "inventario":        (silver_inventario, {"fecha": Date(), "cod_sucursal": NVARCHAR(10), "sku": NVARCHAR(20),
                                                  "archivo_origen": NVARCHAR(60), "run_id": NVARCHAR(40),
                                                  "fecha_carga": DateTime()}),
        "cuarentena_inventario": (cuarentena_inventario, tipos_texto(cuarentena_inventario)),
        "reglas_calidad":    (reglas_calidad, {"tabla": NVARCHAR(20), "regla": NVARCHAR(60), "dimension_dama": NVARCHAR(30),
                                               "accion": NVARCHAR(80), "run_id": NVARCHAR(40)}),
        "productos":         (silver_productos.assign(**lin), None),
        "sucursales":        (silver_sucursales.assign(**lin), None),
        "clientes":          (silver_clientes.assign(**lin), {"id_cliente": NVARCHAR(64)}),
        "campanas":          (silver_campanas.assign(**lin), None),
    }, "SILVER")
except Exception:
    fallar("SILVER")
    raise

## 8. Capa Gold: dimensiones

| Dimensión | Tratamiento |
|---|---|
| `dim_fecha` | Calendario del periodo · jerarquía Día → Mes → Trimestre → Año |
| `dim_geografia` | Copo de nieve compartido por Sucursal y Cliente · Comuna → Región → País |
| `dim_producto` | **SCD 2** en `precio_lista` y `categoria` · **SCD 1** en `marca` (y demás atributos) |
| `dim_cliente` | **SCD 2** en `segmento` y comuna/región (`geografia_key`) |
| `dim_sucursal` | **SCD 3** en formato: `formato_actual`, `formato_anterior`, `fecha_cambio_formato` |
| `dim_canal`, `dim_campana` | Estáticas |

Todas las dimensiones tienen una fila **−1** (miembro desconocido) para que ninguna venta se pierda en los JOIN.

In [ ]:
lg = log("GOLD")
try:
    # --- DIM_FECHA ----------------------------------------------------------
    DIAS  = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"]
    MESES = ["Enero", "Febrero", "Marzo", "Abril", "Mayo", "Junio", "Julio", "Agosto",
             "Septiembre", "Octubre", "Noviembre", "Diciembre"]
    cal = pd.date_range(FECHA_INICIO_PERIODO, FECHA_FIN_PERIODO, freq="D")
    dim_fecha = pd.DataFrame({
        "fecha_key":        cal.strftime("%Y%m%d").astype(int),
        "fecha":            cal.strftime("%Y-%m-%d"),
        "dia":              cal.day,
        "nombre_dia":       [DIAS[d] for d in cal.day_of_week],
        "es_fin_de_semana": cal.day_of_week >= 5,
        "mes":              cal.month,
        "nombre_mes":       [MESES[m - 1] for m in cal.month],
        "trimestre":        cal.quarter,
        "anio":             cal.year,
    })
    dim_fecha = pd.concat([pd.DataFrame([{"fecha_key": -1, "fecha": "", "dia": 0, "nombre_dia": "Desconocido",
                                          "es_fin_de_semana": False, "mes": 0, "nombre_mes": "Desconocido",
                                          "trimestre": 0, "anio": 0}]), dim_fecha], ignore_index=True)

    # --- DIM_GEOGRAFIA (copo de nieve) ----------------------------------------
    geo = (pd.concat([silver_sucursales[["comuna", "region"]], silver_clientes[["comuna", "region"]]])
           .drop_duplicates().sort_values(["region", "comuna"]).reset_index(drop=True))
    geo.insert(0, "geografia_key", range(1, len(geo) + 1))
    geo["pais"] = "Chile"
    dim_geografia = pd.concat([pd.DataFrame([{"geografia_key": -1, "comuna": "Desconocida",
                                              "region": "Desconocida", "pais": "Desconocido"}]), geo], ignore_index=True)
    lg.info(f"dim_fecha: {len(dim_fecha)} filas · dim_geografia: {len(dim_geografia)} filas")
except Exception:
    fallar("GOLD")
    raise

### 8.1 SCD Tipo 2 (Producto y Cliente)
Se comparan v1 y v2 **solo en los atributos Tipo 2**:
- Sin cambio en atributos Tipo 2 → **una sola fila** vigente todo el periodo (con los atributos Tipo 1 ya corregidos).
- Con cambio → la versión anterior se cierra el `2025-02-28` (`es_actual = 0`) y se abre una nueva desde el `2025-03-01`.
- Registro nuevo en v2 → una fila vigente desde el `2025-03-01`.

Los atributos **Tipo 1** (por ejemplo la marca `"KUMBRE "` → `"Kumbre"`) se sobrescriben en **todas** las versiones con el valor de la v2.

In [ ]:
def aplicar_scd2(v1, v2, clave, attrs_t2, attrs_t1):
    attrs = attrs_t2 + attrs_t1
    m = v1[[clave] + attrs].merge(v2[[clave] + attrs], on=clave, how="outer",
                                  suffixes=("_v1", "_v2"), indicator=True)
    ambos = m["_merge"].eq("both")
    cambio = pd.Series(False, index=m.index)
    for a in attrs_t2:
        cambio |= ambos & m[f"{a}_v1"].ne(m[f"{a}_v2"])

    def tomar(filas, origen_t2, origen_t1, inicio, fin, actual):
        out = filas[[clave]].copy()
        for a in attrs_t2:
            out[a] = filas[f"{a}_{origen_t2}"]
        for a in attrs_t1:
            out[a] = filas[f"{a}_{origen_t1}"]
        return out.assign(fecha_inicio=inicio, fecha_fin=fin, es_actual=actual)

    partes = [
        tomar(m[ambos & ~cambio], "v2", "v2", FECHA_INICIO_PERIODO, FECHA_ABIERTA, True),   # sin cambio Tipo 2
        tomar(m[cambio], "v1", "v2", FECHA_INICIO_PERIODO, FECHA_FIN_V1, False),           # versión cerrada
        tomar(m[cambio], "v2", "v2", FECHA_V2, FECHA_ABIERTA, True),                       # versión nueva
        tomar(m[m["_merge"].eq("right_only")], "v2", "v2", FECHA_V2, FECHA_ABIERTA, True),  # alta en v2
        tomar(m[m["_merge"].eq("left_only")], "v1", "v1", FECHA_INICIO_PERIODO, FECHA_ABIERTA, True),
    ]
    dim = pd.concat(partes, ignore_index=True).sort_values([clave, "fecha_inicio"]).reset_index(drop=True)
    dim = dim.astype({a: v2[a].dtype for a in attrs})   # el outer merge convierte enteros en float
    detalle = {a: int((ambos & m[f"{a}_v1"].ne(m[f"{a}_v2"])).sum()) for a in attrs}
    return dim, int(cambio.sum()), int(m["_merge"].eq("right_only").sum()), detalle

lg = log("GOLD")
try:
    # --- DIM_PRODUCTO: SCD2 precio_lista y categoria · SCD1 resto -------------
    p1 = silver_productos[silver_productos["version"] == "v1"]
    p2 = silver_productos[silver_productos["version"] == "v2"]
    dim_producto, n_cambios, n_altas, det = aplicar_scd2(
        p1, p2, "sku", attrs_t2=["precio_lista", "categoria"],
        attrs_t1=["producto", "subcategoria", "marca", "costo_unitario"])
    dim_producto.insert(0, "producto_key", range(1, len(dim_producto) + 1))
    lg.info(f"dim_producto SCD2: {n_cambios} SKU con nueva versión, {n_altas} alta(s) · cambios por atributo {det}")
    dim_producto = pd.concat([pd.DataFrame([{
        "producto_key": -1, "sku": "DESCONOCIDO", "precio_lista": 0, "categoria": "Desconocida",
        "producto": "Producto desconocido", "subcategoria": "Desconocida", "marca": "Desconocida",
        "costo_unitario": 0, "fecha_inicio": FECHA_INICIO_PERIODO, "fecha_fin": FECHA_ABIERTA,
        "es_actual": True}]), dim_producto], ignore_index=True)
    dim_producto = dim_producto[["producto_key", "sku", "producto", "subcategoria", "categoria", "marca",
                                 "precio_lista", "costo_unitario", "fecha_inicio", "fecha_fin", "es_actual"]]

    # --- DIM_CLIENTE: SCD2 segmento y comuna/región ----------------------------
    c1 = silver_clientes[silver_clientes["version"] == "v1"]
    c2 = silver_clientes[silver_clientes["version"] == "v2"]
    dim_cliente, n_cambios, n_altas, det = aplicar_scd2(
        c1, c2, "id_cliente", attrs_t2=["segmento", "comuna", "region"], attrs_t1=["fecha_registro"])
    dim_cliente = dim_cliente.merge(dim_geografia[["geografia_key", "comuna", "region"]],
                                    on=["comuna", "region"], how="left")
    dim_cliente["geografia_key"] = dim_cliente["geografia_key"].fillna(-1).astype(int)
    dim_cliente = dim_cliente.sort_values(["id_cliente", "fecha_inicio"]).reset_index(drop=True)
    dim_cliente.insert(0, "cliente_key", range(1, len(dim_cliente) + 1))
    lg.info(f"dim_cliente SCD2: {n_cambios:,} clientes con nueva versión, {n_altas:,} altas · cambios por atributo {det}")
    dim_cliente = pd.concat([pd.DataFrame([{
        "cliente_key": -1, "id_cliente": "NO_IDENTIFICADO", "segmento": "No identificado",
        "geografia_key": -1, "fecha_registro": "", "fecha_inicio": FECHA_INICIO_PERIODO,
        "fecha_fin": FECHA_ABIERTA, "es_actual": True}]), dim_cliente], ignore_index=True)
    dim_cliente = dim_cliente[["cliente_key", "id_cliente", "segmento", "geografia_key", "fecha_registro",
                               "fecha_inicio", "fecha_fin", "es_actual"]]
    lg.info(f"dim_producto: {len(dim_producto)} filas · dim_cliente: {len(dim_cliente):,} filas (incluye fila −1)")
except Exception:
    fallar("GOLD")
    raise

### 8.2 SCD Tipo 3 (Sucursal) y dimensiones estáticas
Una sola fila por sucursal. Si el formato cambió entre v1 y v2, el valor previo pasa a `formato_anterior` y se registra `fecha_cambio_formato`.

In [ ]:
lg = log("GOLD")
try:
    s1 = silver_sucursales[silver_sucursales["version"] == "v1"][["cod_sucursal", "formato"]]
    s2 = silver_sucursales[silver_sucursales["version"] == "v2"]
    suc = s2.merge(s1.rename(columns={"formato": "formato_v1"}), on="cod_sucursal", how="left")
    cambio = suc["formato_v1"].notna() & suc["formato"].ne(suc["formato_v1"])
    suc["formato_actual"] = suc["formato"]
    suc["formato_anterior"] = suc["formato_v1"].where(cambio, None)
    suc["fecha_cambio_formato"] = np.where(cambio, FECHA_V2, "")
    suc = suc.merge(dim_geografia[["geografia_key", "comuna", "region"]], on=["comuna", "region"], how="left")
    suc = suc.sort_values("cod_sucursal").reset_index(drop=True)
    suc.insert(0, "sucursal_key", range(1, len(suc) + 1))
    cols_suc = ["sucursal_key", "cod_sucursal", "sucursal", "formato_actual", "formato_anterior",
                "fecha_cambio_formato", "latitud", "longitud", "fecha_apertura", "geografia_key"]
    dim_sucursal = pd.concat([pd.DataFrame([{
        "sucursal_key": -1, "cod_sucursal": "S00", "sucursal": "Desconocida", "formato_actual": "Desconocido",
        "formato_anterior": None, "fecha_cambio_formato": "", "latitud": None, "longitud": None,
        "fecha_apertura": "", "geografia_key": -1}]), suc[cols_suc]], ignore_index=True)
    lg.info(f"dim_sucursal SCD3: {int(cambio.sum())} sucursales con cambio de formato: "
            f"{', '.join(suc.loc[cambio, 'sucursal'])}")

    dim_canal = pd.DataFrame({
        "canal_key":  [-1, 1, 2, 3],
        "canal":      ["Desconocido", "Tienda física", "E-commerce", "App"],
        "tipo_canal": ["Desconocido", "Presencial", "Digital", "Digital"],
    })

    camp = silver_campanas[silver_campanas["version"] == "v2"].reset_index(drop=True)
    camp.insert(0, "campana_key", range(1, len(camp) + 1))
    dim_campana = pd.concat([pd.DataFrame([{
        "campana_key": -1, "cod_campana": "", "campana": "Sin campaña", "tipo_campana": "Sin campaña",
        "pct_descuento": 0.0, "fecha_inicio": "", "fecha_fin": ""}]),
        camp[["campana_key", "cod_campana", "campana", "tipo_campana", "pct_descuento", "fecha_inicio", "fecha_fin"]]],
        ignore_index=True)
    lg.info(f"dim_sucursal: {len(dim_sucursal)} · dim_canal: {len(dim_canal)} · dim_campana: {len(dim_campana)} filas")
except Exception:
    fallar("GOLD")
    raise
display(dim_sucursal)

## 9. Capa Gold: tabla de hechos `fact_ventas`
Grano: **una fila por línea de boleta** (`nro_boleta` + `nro_linea`).
Para Producto y Cliente (SCD 2) se busca la versión cuya vigencia contiene la fecha de la venta (`fecha_inicio ≤ fecha ≤ fecha_fin`). Lo que no encuentra correspondencia toma la clave **−1**: ninguna fila se pierde.

| Medida | Fórmula |
|---|---|
| `monto_bruto` | `precio_unitario × cantidad` |
| `cantidad_devuelta` | `cantidad` si `estado = DEVUELTA`, si no 0 |
| `monto_devuelto` | `monto_bruto − descuento` de las líneas devueltas |
| `monto_neto` | `monto_bruto − descuento − monto_devuelto` |
| `costo_total` | `costo_unitario × cantidad` |
| `margen` | `monto_neto − costo_total` |

In [ ]:
def buscar_vigente(hechos, dim, col_hecho, col_dim, clave, id_col="id_venta"):
    """Devuelve la clave de la versión vigente en la fecha de cada fila (−1 si no hay)."""
    cand = hechos[[id_col, col_hecho, "fecha_txt"]].merge(
        dim.loc[dim[clave] != -1, [col_dim, clave, "fecha_inicio", "fecha_fin"]],
        left_on=col_hecho, right_on=col_dim, how="inner")
    cand = cand[(cand["fecha_txt"] >= cand["fecha_inicio"]) & (cand["fecha_txt"] <= cand["fecha_fin"])]
    return hechos[id_col].map(cand.set_index(id_col)[clave]).fillna(-1).astype(int)

lg = log("GOLD")
try:
    f = silver_ventas.copy()
    f["fecha_txt"] = f["fecha"].dt.strftime("%Y-%m-%d")
    f["fecha_key"] = f["fecha"].dt.strftime("%Y%m%d").astype(int)
    f["canal_key"]    = f["canal"].map(dim_canal.set_index("canal")["canal_key"]).fillna(-1).astype(int)
    f["sucursal_key"] = f["sucursal"].map(dim_sucursal[dim_sucursal["sucursal_key"] != -1]
                                          .set_index("sucursal")["sucursal_key"]).fillna(-1).astype(int)
    f["campana_key"]  = f["cod_campana"].map(dim_campana[dim_campana["campana_key"] != -1]
                                             .set_index("cod_campana")["campana_key"]).fillna(-1).astype(int)
    f["producto_key"] = buscar_vigente(f, dim_producto, "sku", "sku", "producto_key")
    f["cliente_key"]  = buscar_vigente(f, dim_cliente, "id_cliente", "id_cliente", "cliente_key")

    # Validación de exactitud: el precio de la venta debe ser el precio_lista de la versión vigente
    prod = dim_producto.set_index("producto_key")
    precio_dim = f["producto_key"].map(prod["precio_lista"])
    distintos = ((f["producto_key"] != -1) & (precio_dim != f["precio_unitario"])).sum()
    (lg.warning if distintos else lg.info)(f"Precio de venta vs precio_lista vigente (SCD2): {distintos} diferencias")

    # Medidas
    devuelta = f["estado"].eq("DEVUELTA")
    f["monto_bruto"]       = f["precio_unitario"] * f["cantidad"]
    f["cantidad_devuelta"] = np.where(devuelta, f["cantidad"], 0)
    f["monto_devuelto"]    = np.where(devuelta, f["monto_bruto"] - f["descuento"], 0)
    f["monto_neto"]        = f["monto_bruto"] - f["descuento"] - f["monto_devuelto"]
    f["costo_total"]       = f["producto_key"].map(prod["costo_unitario"]).astype("int64") * f["cantidad"]
    f["margen"]            = f["monto_neto"] - f["costo_total"]

    cols_fact = ["id_venta", "nro_boleta", "nro_linea",
                 "fecha_key", "producto_key", "cliente_key", "sucursal_key", "canal_key", "campana_key",
                 "cantidad", "precio_unitario", "monto_bruto", "descuento", "cantidad_devuelta",
                 "monto_devuelto", "monto_neto", "costo_total", "margen",
                 "archivo_origen", "run_id", "fecha_carga"]
    fact_ventas = f[cols_fact].copy()

    for k in ["producto_key", "cliente_key", "sucursal_key", "canal_key", "campana_key"]:
        lg.info(f"fact_ventas.{k}: {(fact_ventas[k] == -1).sum():,} filas con clave −1")
    lg.info(f"fact_ventas: {len(fact_ventas):,} filas (Silver: {len(silver_ventas):,})")
except Exception:
    fallar("GOLD")
    raise
fact_ventas.head()

### 9.1 Capa Gold: tabla de hechos `fact_inventario_diario`
Grano: **una fila por día, sucursal y producto** (foto del stock al cierre). `id_inventario` es una clave sustituta.
El producto se busca con la misma regla de vigencia SCD 2 que en ventas, y `valor_inventario` usa el costo de la versión vigente ese día.

| Medida | Tipo | Cómo se agrega |
|---|---|---|
| `cantidad_stock` | **Semi-aditiva** | Se suma entre productos y sucursales; entre días se usa el promedio o el último valor |
| `stock_minimo` | No aditiva | Parámetro de reposición; se compara, no se suma |
| `valor_inventario` = `cantidad_stock × costo_unitario` | **Semi-aditiva** | Igual que `cantidad_stock` |

In [ ]:
lg = log("GOLD")
try:
    fi = silver_inventario.copy()
    fi["id_inventario"] = np.arange(1, len(fi) + 1)
    fi["fecha_txt"] = fi["fecha"].dt.strftime("%Y-%m-%d")
    fi["fecha_key"] = fi["fecha"].dt.strftime("%Y%m%d").astype(int)
    fi["sucursal_key"] = fi["cod_sucursal"].map(dim_sucursal[dim_sucursal["sucursal_key"] != -1]
                                                .set_index("cod_sucursal")["sucursal_key"]).fillna(-1).astype(int)
    fi["producto_key"] = buscar_vigente(fi, dim_producto, "sku", "sku", "producto_key", id_col="id_inventario")
    fi["valor_inventario"] = fi["cantidad_stock"] * fi["producto_key"].map(dim_producto.set_index("producto_key")["costo_unitario"]).astype("int64")

    fact_inventario_diario = fi[["id_inventario", "fecha_key", "producto_key", "sucursal_key",
                                 "cantidad_stock", "stock_minimo", "valor_inventario",
                                 "archivo_origen", "run_id", "fecha_carga"]].copy()
    bajo = (fact_inventario_diario["cantidad_stock"] < fact_inventario_diario["stock_minimo"]).sum()
    lg.info(f"fact_inventario_diario: {len(fact_inventario_diario):,} filas · {bajo:,} bajo stock mínimo · "
            f"claves −1: producto {(fi['producto_key'] == -1).sum()}, sucursal {(fi['sucursal_key'] == -1).sum()}")
except Exception:
    fallar("GOLD")
    raise
fact_inventario_diario.head()

## 10. Capa Gold: data marts de Marketing y Finanzas
Se construyen desde `fact_ventas` en la misma ejecución, por lo que sus totales de `monto_neto` cuadran con ella.

| Data mart | Tabla de hechos | Grano |
|---|---|---|
| Ventas | `fact_ventas` | Una línea de boleta |
| Marketing | `fact_campana_diaria` | Día × campaña × canal × sucursal |
| Finanzas | `fact_resultado_mensual` | Mes × producto × sucursal (`fecha_key` = primer día del mes) |

In [ ]:
lg = log("GOLD")
try:
    fact_campana_diaria = (fact_ventas
        .groupby(["fecha_key", "campana_key", "canal_key", "sucursal_key"], as_index=False)
        .agg(lineas_vendidas=("id_venta", "count"), unidades=("cantidad", "sum"),
             monto_bruto=("monto_bruto", "sum"), descuento_total=("descuento", "sum"),
             monto_neto=("monto_neto", "sum"))
        .assign(run_id=RUN_ID, fecha_carga=FECHA_CARGA))

    fact_resultado_mensual = (fact_ventas
        .assign(fecha_key=(fact_ventas["fecha_key"] // 100) * 100 + 1)
        .groupby(["fecha_key", "producto_key", "sucursal_key"], as_index=False)
        .agg(ingreso_neto=("monto_neto", "sum"), costo_total=("costo_total", "sum"),
             margen=("margen", "sum"), cantidad_devuelta=("cantidad_devuelta", "sum"),
             monto_devuelto=("monto_devuelto", "sum"))
        .assign(run_id=RUN_ID, fecha_carga=FECHA_CARGA))
    lg.info(f"fact_campana_diaria: {len(fact_campana_diaria):,} filas · "
            f"fact_resultado_mensual: {len(fact_resultado_mensual):,} filas")
except Exception:
    fallar("GOLD")
    raise

## 11. Carga de Gold en SQL Server, claves primarias y foráneas
Las tablas se reemplazan en orden (primero hechos, luego dimensiones) y después se crean las PK y FK para que SQL Server garantice la integridad referencial.

In [ ]:
MONTO = Numeric(16, 2)
T_FECHAS = {"fecha_inicio": Date(), "fecha_fin": Date()}
GOLD = {
    "dim_fecha":     (dim_fecha.assign(fecha=a_fecha(dim_fecha["fecha"])),
                      {"fecha_key": Integer(), "fecha": Date(), "nombre_dia": NVARCHAR(15),
                       "es_fin_de_semana": Boolean(), "nombre_mes": NVARCHAR(15)}),
    "dim_geografia": (dim_geografia, {"geografia_key": Integer(), "comuna": NVARCHAR(50),
                                      "region": NVARCHAR(50), "pais": NVARCHAR(30)}),
    "dim_producto":  (dim_producto.assign(fecha_inicio=a_fecha(dim_producto["fecha_inicio"]),
                                          fecha_fin=a_fecha(dim_producto["fecha_fin"])),
                      {"producto_key": Integer(), "sku": NVARCHAR(20), "producto": NVARCHAR(60),
                       "subcategoria": NVARCHAR(40), "categoria": NVARCHAR(40), "marca": NVARCHAR(40),
                       "precio_lista": MONTO, "costo_unitario": MONTO, "es_actual": Boolean(), **T_FECHAS}),
    "dim_cliente":   (dim_cliente.assign(fecha_registro=a_fecha(dim_cliente["fecha_registro"]),
                                         fecha_inicio=a_fecha(dim_cliente["fecha_inicio"]),
                                         fecha_fin=a_fecha(dim_cliente["fecha_fin"])),
                      {"cliente_key": Integer(), "id_cliente": NVARCHAR(64), "segmento": NVARCHAR(30),
                       "geografia_key": Integer(), "fecha_registro": Date(), "es_actual": Boolean(), **T_FECHAS}),
    "dim_sucursal":  (dim_sucursal.assign(fecha_cambio_formato=a_fecha(dim_sucursal["fecha_cambio_formato"]),
                                          fecha_apertura=a_fecha(dim_sucursal["fecha_apertura"])),
                      {"sucursal_key": Integer(), "cod_sucursal": NVARCHAR(10), "sucursal": NVARCHAR(50),
                       "formato_actual": NVARCHAR(30), "formato_anterior": NVARCHAR(30),
                       "fecha_cambio_formato": Date(), "latitud": Numeric(9, 6), "longitud": Numeric(9, 6),
                       "fecha_apertura": Date(), "geografia_key": Integer()}),
    "dim_canal":     (dim_canal, {"canal_key": Integer(), "canal": NVARCHAR(30), "tipo_canal": NVARCHAR(20)}),
    "dim_campana":   (dim_campana.assign(fecha_inicio=a_fecha(dim_campana["fecha_inicio"]),
                                         fecha_fin=a_fecha(dim_campana["fecha_fin"])),
                      {"campana_key": Integer(), "cod_campana": NVARCHAR(20), "campana": NVARCHAR(60),
                       "tipo_campana": NVARCHAR(30), "pct_descuento": Numeric(5, 2), **T_FECHAS}),
    "fact_ventas":   (fact_ventas,
                      {**{k: Integer() for k in ["id_venta", "nro_boleta", "nro_linea", "fecha_key", "producto_key",
                                                  "cliente_key", "sucursal_key", "canal_key", "campana_key",
                                                  "cantidad", "cantidad_devuelta"]},
                       **{k: MONTO for k in ["precio_unitario", "monto_bruto", "descuento", "monto_devuelto",
                                             "monto_neto", "costo_total", "margen"]},
                       "archivo_origen": NVARCHAR(60), "run_id": NVARCHAR(40), "fecha_carga": DateTime()}),
    "fact_inventario_diario": (fact_inventario_diario,
                      {**{k: Integer() for k in ["id_inventario", "fecha_key", "producto_key", "sucursal_key",
                                                  "cantidad_stock", "stock_minimo"]},
                       "valor_inventario": MONTO, "archivo_origen": NVARCHAR(60), "run_id": NVARCHAR(40),
                       "fecha_carga": DateTime()}),
    "fact_campana_diaria": (fact_campana_diaria,
                      {**{k: Integer() for k in ["fecha_key", "campana_key", "canal_key", "sucursal_key",
                                                  "lineas_vendidas", "unidades"]},
                       **{k: MONTO for k in ["monto_bruto", "descuento_total", "monto_neto"]},
                       "run_id": NVARCHAR(40), "fecha_carga": DateTime()}),
    "fact_resultado_mensual": (fact_resultado_mensual,
                      {**{k: Integer() for k in ["fecha_key", "producto_key", "sucursal_key", "cantidad_devuelta"]},
                       **{k: MONTO for k in ["ingreso_neto", "costo_total", "margen", "monto_devuelto"]},
                       "run_id": NVARCHAR(40), "fecha_carga": DateTime()}),
}

PK = {
    "dim_fecha": ["fecha_key"], "dim_geografia": ["geografia_key"], "dim_producto": ["producto_key"],
    "dim_cliente": ["cliente_key"], "dim_sucursal": ["sucursal_key"], "dim_canal": ["canal_key"],
    "dim_campana": ["campana_key"], "fact_ventas": ["id_venta"], "fact_inventario_diario": ["id_inventario"],
    "fact_campana_diaria": ["fecha_key", "campana_key", "canal_key", "sucursal_key"],
    "fact_resultado_mensual": ["fecha_key", "producto_key", "sucursal_key"],
}
FK = {  # tabla: [(columna, dimensión)]
    "dim_cliente":  [("geografia_key", "dim_geografia")],
    "dim_sucursal": [("geografia_key", "dim_geografia")],
    "fact_ventas":  [("fecha_key", "dim_fecha"), ("producto_key", "dim_producto"), ("cliente_key", "dim_cliente"),
                     ("sucursal_key", "dim_sucursal"), ("canal_key", "dim_canal"), ("campana_key", "dim_campana")],
    "fact_inventario_diario": [("fecha_key", "dim_fecha"), ("producto_key", "dim_producto"),
                               ("sucursal_key", "dim_sucursal")],
    "fact_campana_diaria":    [("fecha_key", "dim_fecha"), ("campana_key", "dim_campana"),
                               ("canal_key", "dim_canal"), ("sucursal_key", "dim_sucursal")],
    "fact_resultado_mensual": [("fecha_key", "dim_fecha"), ("producto_key", "dim_producto"),
                               ("sucursal_key", "dim_sucursal")],
}
ORDEN_BORRADO = ["fact_ventas", "fact_inventario_diario", "fact_campana_diaria", "fact_resultado_mensual",
                 "dim_cliente", "dim_sucursal", "dim_producto", "dim_fecha", "dim_canal", "dim_campana", "dim_geografia"]

def sentencias_claves():
    out = []
    for tabla, cols in PK.items():
        for c in cols:
            out.append(f"ALTER TABLE gold.{tabla} ALTER COLUMN {c} INT NOT NULL")
        out.append(f"ALTER TABLE gold.{tabla} ADD CONSTRAINT pk_{tabla} PRIMARY KEY ({', '.join(cols)})")
    for tabla, fks in FK.items():
        for col, dim in fks:
            out.append(f"ALTER TABLE gold.{tabla} ADD CONSTRAINT fk_{tabla}_{col} "
                       f"FOREIGN KEY ({col}) REFERENCES gold.{dim} ({PK[dim][0]})")
    return out

try:
    escribir_capa("gold", GOLD, "GOLD", borrar_antes=ORDEN_BORRADO)
    ejecutar(sentencias_claves())
    log("GOLD").info(f"Claves creadas: {len(PK)} PK y {sum(len(v) for v in FK.values())} FK")
except Exception:
    fallar("GOLD")
    raise
RESUMEN["filas_gold"] = len(fact_ventas)

## 12. Validación posterior a la carga
Se consulta la base de datos (no los DataFrames) para comprobar que lo cargado cuadra con Silver.

In [ ]:
lg = log("VALIDACION")
validaciones = []
def validar(nombre, esperado, obtenido, nivel="ERROR"):
    ok = esperado == obtenido
    validaciones.append({"validacion": nombre, "esperado": esperado, "obtenido": obtenido,
                         "estado": "OK" if ok else nivel})
    if ok:
        lg.info(f"OK    {nombre}: {obtenido:,}")
    else:
        getattr(lg, nivel.lower())(f"{nivel} {nombre}: esperado {esperado:,}, obtenido {obtenido:,}")

try:
    n = consultar("""SELECT (SELECT COUNT(*) FROM bronze.ventas_raw) AS bronze,
                            (SELECT COUNT(*) FROM silver.ventas) AS silver,
                            (SELECT COUNT(*) FROM silver.cuarentena_ventas) AS cuarentena,
                            (SELECT COUNT(*) FROM gold.fact_ventas) AS gold""").iloc[0]
    validar("Filas Bronze = filas de la fuente", RESUMEN["filas_fuente"], int(n["bronze"]))
    validar("Filas Gold = filas Silver", int(n["silver"]), int(n["gold"]))
    validar("Bronze = Silver + cuarentena + duplicados",
            int(n["bronze"]), int(n["silver"]) + int(n["cuarentena"]) + int(reglas_calidad.query("regla == 'linea_duplicada'")["filas"].sum()))

    for tabla, fks in FK.items():
        for col, dim in fks:
            huerfanas = consultar(f"""SELECT COUNT(*) AS n FROM gold.{tabla} t
                                      LEFT JOIN gold.{dim} d ON t.{col} = d.{PK[dim][0]}
                                      WHERE d.{PK[dim][0]} IS NULL""").iloc[0]["n"]
            validar(f"Claves huérfanas {tabla}.{col}", 0, int(huerfanas))

    sumas = consultar("""SELECT (SELECT SUM(CAST(monto_bruto AS DECIMAL(18,2))) FROM gold.fact_ventas) AS gold_bruto,
                                (SELECT SUM(CAST(monto_neto AS DECIMAL(18,2))) FROM gold.fact_ventas) AS gold_neto,
                                (SELECT SUM(CAST(monto_neto AS DECIMAL(18,2))) FROM gold.fact_campana_diaria) AS mkt_neto,
                                (SELECT SUM(CAST(ingreso_neto AS DECIMAL(18,2))) FROM gold.fact_resultado_mensual) AS fin_neto""").iloc[0]
    silver_bruto = int((silver_ventas["precio_unitario"] * silver_ventas["cantidad"]).sum())
    validar("Suma monto_bruto Gold = Silver", silver_bruto, int(sumas["gold_bruto"]), "WARNING")
    validar("Monto neto Marketing = fact_ventas", int(sumas["gold_neto"]), int(sumas["mkt_neto"]))
    validar("Monto neto Finanzas = fact_ventas", int(sumas["gold_neto"]), int(sumas["fin_neto"]))

    dups = consultar("SELECT COUNT(*) - COUNT(DISTINCT CONCAT(nro_boleta, '-', nro_linea)) AS n FROM gold.fact_ventas").iloc[0]["n"]
    validar("Grano único nro_boleta + nro_linea", 0, int(dups))

    ni = consultar("""SELECT (SELECT COUNT(*) FROM bronze.inventario_raw) AS bronze,
                             (SELECT COUNT(*) FROM silver.inventario) AS silver,
                             (SELECT COUNT(*) FROM silver.cuarentena_inventario) AS cuarentena,
                             (SELECT COUNT(*) FROM gold.fact_inventario_diario) AS gold""").iloc[0]
    dup_inv = int(reglas_calidad.query("tabla == 'inventario' and regla == 'fila_duplicada'")["filas"].sum())
    validar("Inventario: filas Gold = filas Silver", int(ni["silver"]), int(ni["gold"]))
    validar("Inventario: Bronze = Silver + cuarentena + duplicados",
            int(ni["bronze"]), int(ni["silver"]) + int(ni["cuarentena"]) + dup_inv)
    dups_inv = consultar("""SELECT COUNT(*) - COUNT(DISTINCT CONCAT(fecha_key, '-', sucursal_key, '-', producto_key)) AS n
                            FROM gold.fact_inventario_diario""").iloc[0]["n"]
    validar("Inventario: grano único fecha + sucursal + producto", 0, int(dups_inv))
except Exception:
    fallar("VALIDACION")
    raise

resultado_validaciones = pd.DataFrame(validaciones)
errores = resultado_validaciones["estado"].eq("ERROR").sum()
RESUMEN["estado"] = "EXITOSA" if errores == 0 else f"CON {errores} ERRORES DE VALIDACIÓN"
resultado_validaciones

## 13. Registro de auditoría
Cada ejecución deja una fila en `audit.log_ejecuciones` (se acumulan, no se reemplazan).

In [ ]:
RESUMEN["fin"] = datetime.now().replace(microsecond=0)
RESUMEN["archivo_log"] = LOG_FILE.name
auditoria = pd.DataFrame([{k: RESUMEN.get(k) for k in
    ["run_id", "inicio", "fin", "filas_fuente", "filas_bronze", "filas_silver",
     "filas_cuarentena", "filas_gold", "estado", "archivo_log"]}])
try:
    with engine.begin() as con:
        auditoria.to_sql("log_ejecuciones", con=con, schema="audit", if_exists="append", index=False,
                         dtype={"run_id": NVARCHAR(40), "inicio": DateTime(), "fin": DateTime(),
                                "estado": NVARCHAR(60), "archivo_log": NVARCHAR(80)})
except Exception:
    fallar("AUDITORIA")
    raise
log("FIN").info(f"Pipeline {RESUMEN['estado']} · {(RESUMEN['fin'] - RESUMEN['inicio']).seconds}s · log: {LOG_FILE.name}")
auditoria.T

## 14. Consultas de evidencia
Vista rápida de los resultados desde la base de datos. Las consultas completas para SSMS están en `sql/02_consultas_evidencia.sql`.

In [ ]:
display(consultar("SELECT producto_key, sku, producto, categoria, marca, precio_lista, fecha_inicio, fecha_fin, es_actual "
                  "FROM gold.dim_producto WHERE sku IN ('NB-001','SW-011','MS-008') ORDER BY sku, fecha_inicio"))
display(consultar("""SELECT TOP 10 f.fecha, s.sucursal, p.sku, i.cantidad_stock, i.stock_minimo, i.valor_inventario
                     FROM gold.fact_inventario_diario i
                     JOIN gold.dim_fecha f    ON i.fecha_key = f.fecha_key
                     JOIN gold.dim_sucursal s ON i.sucursal_key = s.sucursal_key
                     JOIN gold.dim_producto p ON i.producto_key = p.producto_key
                     WHERE i.cantidad_stock < i.stock_minimo
                     ORDER BY f.fecha, s.sucursal"""))
display(consultar("SELECT sucursal_key, cod_sucursal, sucursal, formato_actual, formato_anterior, fecha_cambio_formato "
                  "FROM gold.dim_sucursal ORDER BY sucursal_key"))
display(consultar("""SELECT f.anio, f.nombre_mes, g.region, p.categoria,
                            SUM(v.monto_neto) AS monto_neto, SUM(v.margen) AS margen
                     FROM gold.fact_ventas v
                     JOIN gold.dim_fecha f     ON v.fecha_key = f.fecha_key
                     JOIN gold.dim_sucursal s  ON v.sucursal_key = s.sucursal_key
                     JOIN gold.dim_geografia g ON s.geografia_key = g.geografia_key
                     JOIN gold.dim_producto p  ON v.producto_key = p.producto_key
                     GROUP BY f.anio, f.mes, f.nombre_mes, g.region, p.categoria
                     ORDER BY f.mes, g.region, p.categoria""").head(15))